# Chapter 19 examples: PILCO

This notebook runs the examples of
[Chapter 19](https://thduynguyen.github.io/gtsam/chapter19): a small version
of PILCO in numpy. The dynamics factor is a Gaussian process, the forward
message is kept Gaussian by moment matching, and the expected return is then
a closed-form function of the policy parameter.

GTSAM Copyright 2010, Georgia Tech Research Corporation,
Atlanta, Georgia 30332-0415
All Rights Reserved

Authors: Frank Dellaert, et al. (see THANKS for the full author list)

See LICENSE for the license information

<a href="https://colab.research.google.com/github/thduynguyen/gtsam/blob/feature/semiringfactor/gtsam/semiring/doc/chapter19_examples.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
try:
    import google.colab
    # This notebook needs only numpy, scipy and plotly, which Colab provides.
    %pip install --quiet numpy scipy plotly
except ImportError:
    pass  # Not running on Colab, do nothing

In [2]:
import numpy as np
from scipy.optimize import minimize, minimize_scalar

np.set_printoptions(precision=3, suppress=True)

## The robot on a hill (Section 1)

The origin is the top of a hill. The slope pushes the robot away from it:
$x' = x + u + \sin x + w$, with $w \sim N(0, 0.1)$. The robot starts at
$x_0 \sim N(2, 1)$, makes four moves with the policy $u = -K x$, and is
rewarded as on the line: $-(x^2 + u^2)$ per move and $-x_4^2$ at the end.

In [3]:
MOVES, NOISE = 4, 0.1
START_MEAN, START_VARIANCE = 2.0, 1.0


def hill(x, u):
    """The mean of the next position."""
    return x + u + np.sin(x)


def collect(rng, episodes, gain=None, action_variance=4.0):
    """Run episodes on the real hill; return inputs (x, u) and changes x' - x.

    With gain=None the actions are random; otherwise u = -gain * x.
    """
    inputs, changes = [], []
    for _ in range(episodes):
        x = START_MEAN + np.sqrt(START_VARIANCE) * rng.normal()
        for t in range(MOVES):
            u = (np.sqrt(action_variance) * rng.normal() if gain is None
                 else -gain * x)
            x_next = hill(x, u) + np.sqrt(NOISE) * rng.normal()
            inputs.append([x, u]), changes.append(x_next - x)
            x = x_next
    return np.array(inputs), np.array(changes)


def true_rollout(gain, samples=200000, seed=100):
    """Monte Carlo on the real hill: J and the moments of x_t."""
    rng = np.random.default_rng(seed)
    x = START_MEAN + np.sqrt(START_VARIANCE) * rng.normal(size=samples)
    total = np.zeros(samples)
    moments = [(x.mean(), x.var())]
    for t in range(MOVES):
        u = -gain * x
        total -= x ** 2 + u ** 2
        x = hill(x, u) + np.sqrt(NOISE) * rng.normal(size=samples)
        moments.append((x.mean(), x.var()))
    return (total - x ** 2).mean(), np.array(moments)


for gain in [0.5, 1.0]:
    J, moments = true_rollout(gain)
    print(f"K = {gain}: true J = {J:.3f}, mean of x_t = {moments[:, 0]}")

K = 0.5: true J = -20.734, mean of x_t = [2.002 1.552 1.636 1.679 1.702]
K = 1.0: true J = -13.762, mean of x_t = [2.002 0.551 0.46  0.394 0.34 ]


## A Gaussian-process dynamics factor (Section 2)

The model predicts the change $x' - x$ from the input $z = (x, u)$, with a
squared-exponential kernel.

In [4]:
class GaussianProcess:
    """GP regression with a squared-exponential kernel and fixed settings."""

    def __init__(self, inputs, targets, lengths, signal, noise):
        self.inputs, self.signal, self.noise = inputs, signal, noise
        self.inverse_lengths = 1 / np.asarray(lengths) ** 2  # Lambda^-1
        gram = self.kernel(inputs, inputs)
        self.solve = np.linalg.inv(gram + noise * np.eye(len(inputs)))
        self.weights = self.solve @ targets  # xi

    def kernel(self, a, b):
        d = a[:, None, :] - b[None, :, :]
        return self.signal * np.exp(
            -0.5 * np.einsum("ijk,k,ijk->ij", d, self.inverse_lengths, d))

    def predict(self, queries):
        """Mean and variance of the function at certain inputs."""
        k = self.kernel(queries, self.inputs)
        variance = self.signal - np.einsum("ij,jk,ik->i", k, self.solve, k)
        return k @ self.weights, variance

    def moments(self, mean, covariance):
        """Moment matching: the input is uncertain, z ~ N(mean, covariance).

        Returns the mean and variance of the predicted change (including the
        noise of the dynamics) and its covariance with the input.
        """
        dimension = len(mean)
        lengths = np.diag(1 / self.inverse_lengths)  # Lambda
        d = self.inputs - mean
        inverse = np.linalg.inv(covariance + lengths)
        # k_bar_i = E[k(z, z_i)]
        k_bar = self.signal / np.sqrt(np.linalg.det(
            covariance * self.inverse_lengths[None, :] + np.eye(dimension))
        ) * np.exp(-0.5 * np.einsum("ij,jk,ik->i", d, inverse, d))
        predicted_mean = self.weights @ k_bar
        cross = covariance @ inverse @ (d.T @ (self.weights * k_bar))
        # Xi_ij = E[k(z, z_i) k(z, z_j)]
        centre = 0.5 * (self.inputs[:, None, :] + self.inputs[None, :, :]) - mean
        apart = self.inputs[:, None, :] - self.inputs[None, :, :]
        inverse2 = np.linalg.inv(covariance + 0.5 * lengths)
        second = self.signal ** 2 / np.sqrt(np.linalg.det(
            2 * covariance * self.inverse_lengths[None, :] + np.eye(dimension))
        ) * np.exp(-0.25 * np.einsum(
            "ijk,k,ijk->ij", apart, self.inverse_lengths, apart)
        ) * np.exp(-0.5 * np.einsum("ijk,kl,ijl->ij", centre, inverse2, centre))
        variance = (self.signal - np.trace(self.solve @ second) +
                    self.weights @ second @ self.weights -
                    predicted_mean ** 2 + self.noise)
        return predicted_mean, variance, cross


LENGTHS, SIGNAL = [1.5, 3.0], 4.0
rng = np.random.default_rng(0)
inputs, changes = collect(rng, 5)
model = GaussianProcess(inputs, changes, LENGTHS, SIGNAL, NOISE)
print(len(changes), "transitions with random actions")
for x in [0.0, 1.0, 2.0, 3.0]:
    mean, variance = model.predict(np.array([[x, -x]]))
    print(f"at x = {x}, u = {-x}: model {mean[0]:6.3f} +- "
          f"{np.sqrt(variance[0]):.3f},  true {hill(x, -x) - x:6.3f}")

20 transitions with random actions
at x = 0.0, u = -0.0: model  0.162 +- 0.226,  true  0.000
at x = 1.0, u = -1.0: model -0.085 +- 0.171,  true -0.159
at x = 2.0, u = -2.0: model -0.826 +- 0.585,  true -1.091
at x = 3.0, u = -3.0: model -1.180 +- 1.348,  true -2.859


## Moment matching, checked by sampling (Section 3)

For a Gaussian input, the mean and variance of the prediction have closed
forms. They agree with the moments of many sampled inputs pushed through the
model.

In [5]:
direction = np.array([1.0, -0.5])  # z = (x, u) = x * (1, -K) with K = 0.5
mean, covariance = 2.0 * direction, 1.0 * np.outer(direction, direction)
matched = model.moments(mean, covariance)

rng = np.random.default_rng(1)
x = 2.0 + rng.normal(size=400000)
sampled_mean, sampled_variance = model.predict(np.outer(x, direction))
sampled = (sampled_mean.mean(),
           sampled_variance.mean() + sampled_mean.var() + NOISE,
           np.cov(x, sampled_mean)[0, 1])
print("moment matching: mean %.4f  variance %.4f  cov(x, change) %.4f"
      % (matched[0], matched[1], matched[2][0]))
print("sampling:        mean %.4f  variance %.4f  cov(x, change) %.4f"
      % sampled)
assert np.allclose([matched[0], matched[1], matched[2][0]], sampled, atol=5e-3)

moment matching: mean -0.0699  variance 0.6718  cov(x, change) -0.4225
sampling:        mean -0.0684  variance 0.6687  cov(x, change) -0.4202


## Stage 1: the forward message, step by step (Section 3)

The state stays Gaussian, $x_t \sim N(\mu_t, \Sigma_t)$, and the expected
return is a sum of closed-form terms.

In [6]:
def stage1(moments_of_change, gain, moves=MOVES, mean=START_MEAN,
           variance=START_VARIANCE):
    """Propagate N(mean, variance) and add up the expected rewards.

    moments_of_change(mean_z, covariance_z) returns the mean and variance of
    x' - x and its covariance with z = (x, u), for a Gaussian z.
    """
    direction = np.array([1.0, -gain])  # z = x * (1, -K)
    total, messages = 0.0, [(mean, variance)]
    for t in range(moves):
        total -= (1 + gain ** 2) * (mean ** 2 + variance)  # E[x^2 + u^2]
        change_mean, change_variance, cross = moments_of_change(
            mean * direction, variance * np.outer(direction, direction))
        mean = mean + change_mean
        variance = variance + change_variance + 2 * cross[0]
        messages.append((mean, variance))
    return total - (mean ** 2 + variance), np.array(messages)

## An exact special case: the line (Section 5)

For linear dynamics the next state of a Gaussian state is exactly Gaussian,
so moment matching is exact. With the true linear model of the line,
$x' = x + u + w$, $w \sim N(0, 0.5)$, two moves and $u = -0.5 x$, it must
return the $J = -9.375$ of Chapter 1.

In [7]:
def linear_moments(F, B, noise):
    """Moments of the change x' - x = (F - 1) x + B u + w."""
    coefficients = np.array([F - 1.0, B])

    def moments(mean, covariance):
        return (coefficients @ mean,
                coefficients @ covariance @ coefficients + noise,
                covariance @ coefficients)

    return moments


J_line, messages_line = stage1(linear_moments(1.0, 1.0, 0.5), 0.5, moves=2)
print("the line, linear model:  J =", J_line)
print("forward messages (mean, variance):\n", messages_line)
assert np.isclose(J_line, -9.375)

# The same with a Gaussian process learned from 300 transitions of the line.
# This is no longer exact: the model itself has an error.
rng = np.random.default_rng(2)
line_x = 2.0 + rng.normal(size=300) * 1.5
line_u = 2.0 * rng.normal(size=300)
line_change = line_u + np.sqrt(0.5) * rng.normal(size=300)
line_model = GaussianProcess(np.stack([line_x, line_u], axis=1), line_change,
                             [6.0, 6.0], 9.0, 0.5)
J_line_gp, _ = stage1(line_model.moments, 0.5, moves=2)
print("the line, Gaussian-process model:  J =", J_line_gp)
assert abs(J_line_gp + 9.375) < 0.3

the line, linear model:  J = -9.375
forward messages (mean, variance):
 [[2.    1.   ]
 [1.    0.75 ]
 [0.5   0.688]]
the line, Gaussian-process model:  J = -9.233848326419222


## Where moment matching errs (Section 6)

To separate the two sources of error, first use the *true* hill in place of
the model. The moments of $\sin x$ under a Gaussian have closed forms, so
the matched moments of one step are exact. The error that remains comes
from assuming that the state is still Gaussian after a nonlinear step.

In [8]:
def hill_moments(mean, covariance):
    """Exact moments of the change u + sin(x) + w for a Gaussian z = (x, u)."""
    mu, var = mean[0], covariance[0, 0]
    damp = np.exp(-var / 2)
    sine = damp * np.sin(mu)  # E[sin x]
    sine_squared = 0.5 * (1 - np.exp(-2 * var) * np.cos(2 * mu))  # E[sin^2 x]
    cov_x_sine = var * damp * np.cos(mu)  # Cov[x, sin x]
    cov_u_sine = covariance[0, 1] / var * cov_x_sine if var > 0 else 0.0
    change_mean = mean[1] + sine
    change_variance = (covariance[1, 1] + sine_squared - sine ** 2 +
                       2 * cov_u_sine + NOISE)
    cross = covariance[:, 1] + np.array([cov_x_sine, cov_u_sine])
    return change_mean, change_variance, cross


for gain in [0.5, 1.0]:
    J_matched, matched = stage1(hill_moments, gain)
    J_true, true = true_rollout(gain)
    print(f"K = {gain}:  J matched {J_matched:.3f}   J true {J_true:.3f}")
    print("   step:             ", np.arange(MOVES + 1))
    print("   mean, matched:    ", matched[:, 0])
    print("   mean, true:       ", true[:, 0])
    print("   variance, matched:", matched[:, 1])
    print("   variance, true:   ", true[:, 1])
    # The first step is exact; later steps are not.
    assert np.allclose(matched[1], true[1], atol=0.01)
J_weak, matched_weak = stage1(hill_moments, 0.5)
_, true_weak = true_rollout(0.5)
assert true_weak[4, 1] > 2 * matched_weak[4, 1]

K = 0.5:  J matched -20.314   J true -20.734
   step:              [0 1 2 3 4]
   mean, matched:     [2.    1.552 1.62  1.7   1.761]
   mean, true:        [2.002 1.552 1.636 1.679 1.702]
   variance, matched: [1.    0.338 0.231 0.169 0.137]
   variance, true:    [0.998 0.337 0.35  0.365 0.384]
K = 1.0:  J matched -13.497   J true -13.762
   step:              [0 1 2 3 4]
   mean, matched:     [2.    0.552 0.442 0.37  0.313]
   mean, true:        [2.002 0.551 0.46  0.394 0.34 ]
   variance, matched: [1.    0.34  0.29  0.286 0.293]
   variance, true:    [0.998 0.341 0.322 0.32  0.325]


## The loop: collect, learn, Stage 1, Stage 2 (Sections 4 and 7)

Stage 2 maximizes the closed-form $J(K)$ of the model with a quasi-Newton
optimizer. Each round then runs the new policy on the real hill for three
episodes and refits the model.

In [9]:
best = minimize_scalar(lambda gain: -true_rollout(gain)[0],
                       bounds=(0, 2), method="bounded")
print(f"best gain on the real hill: K = {best.x:.3f}, J = {-best.fun:.3f}")

rng = np.random.default_rng(0)
inputs, changes = collect(rng, 5)
gain, history = 0.0, []
for round_ in range(5):
    model = GaussianProcess(inputs, changes, LENGTHS, SIGNAL, NOISE)  # learn
    result = minimize(lambda g: -stage1(model.moments, g[0])[0], [gain],
                      method="BFGS")  # stage 2, around stage 1
    gain, predicted = float(result.x[0]), -result.fun
    true = true_rollout(gain)[0]
    history.append((len(changes), gain, predicted, true))
    print(f"round {round_}: {len(changes):2d} transitions,  K = {gain:.3f},  "
          f"predicted J = {predicted:7.3f},  true J = {true:7.3f}")
    new_inputs, new_changes = collect(rng, 3, gain=gain)  # collect
    inputs = np.vstack([inputs, new_inputs])
    changes = np.concatenate([changes, new_changes])
assert abs(history[-1][3] - (-best.fun)) < 0.1
assert history[-1][3] > history[0][3]

best gain on the real hill: K = 0.989, J = -13.758
round 0: 20 transitions,  K = 0.823,  predicted J = -22.962,  true J = -14.619
round 1: 32 transitions,  K = 0.998,  predicted J = -15.970,  true J = -13.760
round 2: 44 transitions,  K = 1.010,  predicted J = -16.597,  true J = -13.771


round 3: 56 transitions,  K = 1.026,  predicted J = -15.769,  true J = -13.797
round 4: 68 transitions,  K = 0.996,  predicted J = -15.134,  true J = -13.760


The forward message of the final model, against the real hill.

In [10]:
J_model, matched = stage1(model.moments, gain)
J_true, true = true_rollout(gain)
print(f"K = {gain:.3f}")
print("mean, model:    ", matched[:, 0])
print("mean, true:     ", true[:, 0])
print("variance, model:", matched[:, 1])
print("variance, true: ", true[:, 1])

K = 0.996
mean, model:     [2.    0.763 0.608 0.493 0.376]
mean, true:      [2.002 0.559 0.468 0.401 0.347]
variance, model: [1.    0.363 0.341 0.386 0.481]
variance, true:  [0.998 0.339 0.322 0.321 0.327]
